# Week 15: Developing Agentic AI — Building Intelligent Fraud Investigation Agents

## Learning Objectives

By the end of this session, you will be able to:
1. **Explain the ReAct pattern** (Reasoning + Acting) and build a simple agent loop from scratch
2. **Create LangChain agents** with custom tools using Amazon Bedrock as the LLM backend
3. **Design custom tools** that agents can call to investigate fraud scenarios
4. **Add conversation memory** so agents remember context across multiple interactions

## Prerequisites

- Completed Weeks 11-14 (LLM APIs, evaluation, Bedrock, fine-tuning)
- Watched pre-class videos on agent architectures, ReAct pattern, LangChain framework, tool calling
- AWS SageMaker notebook environment (Bedrock access via IAM execution role)

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Fraud Investigation Story | 10 min | Code + Markdown |
| Section 1: What Are AI Agents? ReAct from Scratch | 20 min | Demo-heavy |
| Section 2: LangChain Agents + Custom Tools | 25 min | Demo-heavy |
| Lab 1: Build a Fraud Investigation Agent | 15 min | Lab |
| Section 3: Memory & Multi-Turn Conversations | 15 min | Demo |
| Lab 2: Conversational Fraud Analyst | 15 min | Lab |
| Wrap-up & Homework | 5 min | Markdown |

## What We'll Build Today

Over the past weeks, we've been classifying fraud transactions with LLMs — first by
prompting cloud APIs (Weeks 11, 13), then by fine-tuning a local model (Week 14).
But classification is just ONE step in fraud investigation. A real fraud analyst
needs to:

1. **Look up transaction details** in a database
2. **Check the customer's spending history** for anomalies
3. **Calculate a risk score** based on multiple factors
4. **Cross-reference fraud policies** to decide next steps
5. **Explain their reasoning** and recommend an action

Today, we'll build an AI agent that does ALL of this — autonomously reasoning through
a fraud investigation, calling tools as needed, and arriving at a recommendation.

**The big picture**: We're going from "LLM as a calculator" to "LLM as an employee"
— one that can think, use tools, and hold a conversation.

## This Week vs Next Week

This is the first of two weeks on agentic AI. Here's how they fit together:

| | **Week 15 (Today)** | **Week 16 (Next Week)** |
|---|---|---|
| **Focus** | Agent fundamentals — how agents work | Production agents + multi-agent systems |
| **Framework** | LangChain + LangGraph (open-source) | Strands Agents + Bedrock AgentCore (AWS-native) |
| **What you build** | ReAct loop from scratch, single agents with tools and memory | Multi-agent fraud pipeline, persistent memory, production architecture |
| **Key concepts** | ReAct pattern, @tool decorator, MemorySaver | Agents-as-tools pattern, AgentCore Memory, supervisor orchestration |
| **Why it matters** | You need to understand the fundamentals first | Bread Financial uses AgentCore in production — this is what you'll use at work |

**Today** we use LangChain and LangGraph to understand what agents are and how they
work under the hood. **Next week** we pivot to the AWS-native production stack:
**Strands Agents SDK** (simpler API, same concepts) and **Amazon Bedrock AgentCore**
(managed memory, runtime, and gateway). The concepts you learn today — ReAct, tools,
memory — are framework-agnostic. What changes next week is the toolkit, not the thinking.

# Section 0: Environment Setup

We're running on **AWS SageMaker**, which means our notebook already has IAM role-based
access to AWS services — including Amazon Bedrock. No manual credential entry needed!

We'll install LangChain and LangGraph on top of the pre-installed SageMaker environment.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# langchain: Core LangChain framework (chains, prompts, tools)
# langchain-aws: Amazon Bedrock integration (ChatBedrockConverse)
# langgraph: Agent framework with ReAct prebuilt agent
# Note: boto3 is pre-installed on SageMaker — no need to install it

!pip install -q langchain langchain-aws langgraph

# =============================================================================
# IMPORTS
# =============================================================================
import boto3                                      # AWS SDK (pre-installed on SageMaker)
import json                                       # JSON parsing
import os                                         # Environment variables
import importlib.metadata                         # For package version lookup
import sagemaker                                  # SageMaker SDK (pre-installed)
from sagemaker import get_execution_role          # IAM role for Bedrock access

# LangChain imports
from langchain_aws import ChatBedrockConverse     # Bedrock LLM for LangChain
from langchain_core.tools import tool             # @tool decorator
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# LangGraph imports
from langgraph.prebuilt import create_react_agent  # Prebuilt ReAct agent
from langgraph.checkpoint.memory import MemorySaver # Conversation memory

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
import langchain
import langgraph

print("Library versions:")
print(f"  boto3:       {boto3.__version__}")
print(f"  sagemaker:   {sagemaker.__version__}")
print(f"  langchain:   {langchain.__version__}")
print(f"  langgraph:   {importlib.metadata.version('langgraph')}")
print("\n✅ All libraries installed successfully!")

In [ ]:
# =============================================================================
# SAGEMAKER + BEDROCK CONNECTION SETUP
# =============================================================================
# On SageMaker, we get AWS credentials automatically through the execution role.
# No need for getpass or manual credential entry — the IAM role attached to this
# notebook instance already has permissions for Bedrock.

# SageMaker session — gives us the region and execution role
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name  # Typically "us-east-1"

print(f"SageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region: {AWS_REGION}")

# Create Bedrock clients using role-based auth (automatic on SageMaker)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=AWS_REGION)
bedrock_client = boto3.client('bedrock', region_name=AWS_REGION)

# Verify Bedrock connection
try:
    models = bedrock_client.list_foundation_models()
    model_count = len(models['modelSummaries'])
    print(f"\n✅ Connected to Bedrock! {model_count} models available.")
except Exception as e:
    print(f"\n❌ Bedrock connection failed: {e}")
    print("Ask your instructor to verify the execution role has Bedrock permissions.")

# Model we'll use for agents (Claude Haiku — fast, cheap, great at tool use)
BEDROCK_MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"
print(f"\nAgent LLM: {BEDROCK_MODEL_ID}")

In [ ]:
# =============================================================================
# FRAUD INVESTIGATION DATA — Simulated Transaction Database
# =============================================================================
# We'll reuse the same fraud transactions from Weeks 13-14 as our "database"
# that the agent can query. This creates continuity across weeks.
#
# In a real bank, these would live in a database — here we simulate with dicts.

TRANSACTION_DATABASE = {
    "TXN-001": {"id": "TXN-001", "amount": 4500, "merchant": "Unknown Overseas Account", "type": "wire_transfer", "time": "03:47", "location": "International", "description": "Customer reports unauthorized wire transfer of $4,500 to unknown overseas account. No prior international transaction history. Transfer initiated at 3:47 AM local time."},
    "TXN-002": {"id": "TXN-002", "amount": 89.99, "merchant": "Netflix", "type": "subscription", "time": "10:00", "location": "Online", "description": "Regular monthly payment of $89.99 to Netflix streaming service. Consistent with 18-month subscription history. Payment from primary checking account."},
    "TXN-003": {"id": "TXN-003", "amount": 1500, "merchant": "Multiple ATMs", "type": "atm_withdrawal", "time": "14:30", "location": "Multiple cities", "description": "Three consecutive ATM withdrawals totaling $1,500 in different cities within 2 hours. Card was reported lost the following day. Withdrawals at non-bank ATMs."},
    "TXN-004": {"id": "TXN-004", "amount": 234.56, "merchant": "Amazon.com", "type": "online_purchase", "time": "15:20", "location": "Online", "description": "Online purchase of $234.56 at Amazon.com for household electronics. Shipping to address on file. Customer has frequent Amazon purchase history."},
    "TXN-005": {"id": "TXN-005", "amount": 2100, "merchant": "Luxury Jewelry Store", "type": "in_store", "time": "11:30", "location": "Miami", "description": "Customer disputes charge of $2,100 at luxury jewelry store in Miami. Customer's location confirmed as Chicago at time of purchase. No travel alerts set."},
    "TXN-006": {"id": "TXN-006", "amount": 3245.67, "merchant": "ABC Corp", "type": "direct_deposit", "time": "06:00", "location": "N/A", "description": "Automatic payroll direct deposit of $3,245.67 from employer ABC Corp. Matches bi-weekly pay schedule. Amount consistent with employment records."},
    "TXN-007": {"id": "TXN-007", "amount": 87.50, "merchant": "Various Digital Stores", "type": "online_purchase", "time": "22:15", "location": "Online", "description": "Multiple small online purchases ($5-$15) at various digital stores within 30 minutes. None of these merchants appear in customer's history. Different IP addresses used."},
    "TXN-008": {"id": "TXN-008", "amount": 67.23, "merchant": "Whole Foods Market", "type": "in_store", "time": "17:45", "location": "Home area", "description": "Grocery purchase of $67.23 at Whole Foods Market. Customer shops here weekly based on 2-year transaction history. Paid with debit card at POS terminal."},
    "TXN-009": {"id": "TXN-009", "amount": 8200, "merchant": "New Payee Transfer", "type": "wire_transfer", "time": "02:30", "location": "Foreign IP", "description": "Account password changed and $8,200 transferred to a new payee within 15 minutes. Login originated from an IP address in a different country than the account holder's residence."},
    "TXN-010": {"id": "TXN-010", "amount": 3400, "merchant": "Electronics Store Lagos", "type": "in_store", "time": "16:00", "location": "Lagos, Nigeria", "description": "Credit card used for $3,400 purchase at electronics store in Lagos, Nigeria. Cardholder has never traveled outside the United States. Card was not reported stolen."},
}

# Simulated customer spending history — keyed by transaction ID for simplicity
CUSTOMER_HISTORY = {
    "TXN-001": {"avg_monthly_spend": 2500, "international_transactions": 0, "account_age_years": 5, "typical_hours": "8:00-22:00", "flagged_before": False},
    "TXN-002": {"avg_monthly_spend": 3200, "international_transactions": 0, "account_age_years": 3, "typical_hours": "7:00-23:00", "flagged_before": False},
    "TXN-003": {"avg_monthly_spend": 1800, "international_transactions": 2, "account_age_years": 7, "typical_hours": "9:00-21:00", "flagged_before": True},
    "TXN-004": {"avg_monthly_spend": 4100, "international_transactions": 5, "account_age_years": 10, "typical_hours": "6:00-00:00", "flagged_before": False},
    "TXN-005": {"avg_monthly_spend": 3500, "international_transactions": 1, "account_age_years": 4, "typical_hours": "8:00-22:00", "flagged_before": False},
    "TXN-006": {"avg_monthly_spend": 5000, "international_transactions": 3, "account_age_years": 8, "typical_hours": "6:00-23:00", "flagged_before": False},
    "TXN-007": {"avg_monthly_spend": 1200, "international_transactions": 0, "account_age_years": 2, "typical_hours": "9:00-21:00", "flagged_before": False},
    "TXN-008": {"avg_monthly_spend": 2800, "international_transactions": 1, "account_age_years": 6, "typical_hours": "7:00-22:00", "flagged_before": False},
    "TXN-009": {"avg_monthly_spend": 2200, "international_transactions": 0, "account_age_years": 4, "typical_hours": "8:00-20:00", "flagged_before": False},
    "TXN-010": {"avg_monthly_spend": 1500, "international_transactions": 0, "account_age_years": 3, "typical_hours": "9:00-21:00", "flagged_before": False},
}

# Fraud policies — the rules an agent should check against
FRAUD_POLICIES = {
    "international_first_time": "Flag and hold any first-time international transaction over $500. Require customer verification within 24 hours.",
    "unusual_hours": "Transactions between 1:00 AM and 5:00 AM outside customer's typical pattern require enhanced monitoring.",
    "velocity_check": "More than 3 transactions within 30 minutes at different merchants triggers automatic review.",
    "geographic_mismatch": "Transaction location more than 500 miles from customer's last known location within 2 hours requires hold.",
    "amount_threshold": "Single transactions exceeding 3x the customer's average monthly spend require supervisor approval.",
    "new_payee_large_transfer": "Wire transfers over $5,000 to newly added payees require two-factor verification and 24-hour hold.",
    "card_testing_pattern": "Multiple small transactions ($0.01-$5.00) at different merchants within 10 minutes indicate card testing.",
}

print(f"Transaction database: {len(TRANSACTION_DATABASE)} transactions")
print(f"Customer histories:   {len(CUSTOMER_HISTORY)} records")
print(f"Fraud policies:       {len(FRAUD_POLICIES)} rules")
print(f"\nSample transaction IDs: {list(TRANSACTION_DATABASE.keys())[:5]}...")
print(f"\nSample transaction:")
sample = TRANSACTION_DATABASE["TXN-001"]
for key, val in sample.items():
    print(f"  {key}: {val}")

## The Agent Framework Landscape

Before we dive in, let's understand the tools we're using and why:

### This Week: LangChain + LangGraph (Open Source)

- **LangChain** is the most popular open-source agent framework. It's widely used
  in tutorials, blog posts, and prototypes. We teach it first because:
  - Most agent tutorials you'll find online use LangChain
  - It makes the ReAct pattern easy to understand
  - It has excellent Bedrock integration via `langchain-aws`

- **LangGraph** (by the same team) adds stateful workflows and is what
  `create_react_agent` runs on under the hood.

### Next Week: Strands Agents + Amazon Bedrock AgentCore (AWS-Native)

- **Strands Agents SDK** is AWS's open-source agent framework. Think of it as
  "LangChain but simpler and AWS-native." Same concepts (`@tool` decorator,
  system prompts, Bedrock models), cleaner API.

- **Amazon Bedrock AgentCore** is the managed production platform — it provides
  Runtime (hosting), Memory (persistent), Gateway (tool management), Identity
  (credentials), and Observability (monitoring). It's framework-agnostic and
  works with Strands, LangChain, CrewAI, or any framework.

### Why Two Frameworks in Two Weeks?

| Question | Answer |
|----------|--------|
| Why LangChain first? | It's the industry standard for learning. Most resources use it. |
| Why Strands second? | It's what Bread Financial will use in production (AWS-native). |
| Are concepts portable? | YES. `@tool`, ReAct, memory — all the same across frameworks. |
| Is LangChain wasted? | NO. AgentCore can run LangChain agents too. And you'll see LangChain everywhere in the industry. |

**Bottom line**: Learn the PATTERN (ReAct, tools, memory) this week with
LangChain. Apply it next week with the production toolkit (Strands + AgentCore).

# Section 1: What Are AI Agents? The ReAct Pattern

## From LLM to Agent

In Weeks 11-14, we used LLMs as **sophisticated text processors**: give them a
transaction description, get back a classification. The LLM couldn't look anything
up, couldn't calculate anything, couldn't access external data.

An **AI agent** is an LLM that can:
1. **Think** — reason about what information it needs
2. **Act** — call tools to get that information
3. **Observe** — look at the tool results
4. **Repeat** — keep thinking and acting until it has enough to answer

This is the **ReAct** pattern (Reasoning + Acting), introduced by Yao et al. (2023).

## The ReAct Loop

```
User: "Investigate transaction TXN-001"

-> THOUGHT: I need to look up the transaction details first.
-> ACTION: lookup_transaction("TXN-001")
-> OBSERVATION: Wire transfer of $4,500 to unknown overseas account at 3:47 AM...

-> THOUGHT: This is suspicious. Let me check the customer's history.
-> ACTION: check_customer_history("TXN-001")
-> OBSERVATION: No prior international transactions, avg spend $2,500/month...

-> THOUGHT: $4,500 is nearly 2x their monthly average, first international
           transaction, at 3:47 AM. Let me check relevant fraud policies.
-> ACTION: check_fraud_policy("international_first_time")
-> OBSERVATION: Flag and hold any first-time international transaction over $500...

-> THOUGHT: Based on all evidence, this is very likely fraud.
-> FINAL ANSWER: FRAUD — Recommend immediate hold and customer verification.
```

## Why Not Just Prompt the LLM?

You might wonder: "Why not just give the LLM all the data in the prompt?"

| Approach | Pros | Cons |
|----------|------|------|
| **Everything in prompt** | Simple, one API call | Token limits, stale data, can't calculate |
| **Agent with tools** | Dynamic data access, fresh results, can compute | More API calls, harder to debug |

For fraud investigation, agents win because:
- Transaction databases have millions of records (can't fit in a prompt)
- Risk calculations need live computation
- Policies change frequently
- Each investigation needs DIFFERENT data depending on the case

Let's build a ReAct agent from scratch to see exactly how this works.

In [ ]:
# =============================================================================
# DEMO: Build a ReAct Agent from Scratch (No Framework!)
# =============================================================================
# Before using LangChain, let's build a simple ReAct loop with the raw Bedrock
# Converse API. This demystifies what frameworks do under the hood.
#
# We use the bedrock_runtime client from Cell 3 — it already has credentials
# via the SageMaker execution role.

# Step 1: Define our tools as simple Python functions
def scratch_lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID."""
    txn = TRANSACTION_DATABASE.get(transaction_id)
    if txn:
        return json.dumps(txn, indent=2)
    return f"Transaction {transaction_id} not found."

def scratch_check_history(transaction_id: str) -> str:
    """Check customer spending history for a transaction."""
    history = CUSTOMER_HISTORY.get(transaction_id)
    if history:
        return json.dumps(history, indent=2)
    return f"No customer history found for {transaction_id}."

# Step 2: Define tool schemas for Bedrock's toolConfig format
# These tell the LLM what tools are available and how to call them
TOOL_CONFIG = {
    "tools": [
        {
            "toolSpec": {
                "name": "lookup_transaction",
                "description": "Look up transaction details by transaction ID (e.g., TXN-001)",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "transaction_id": {
                                "type": "string",
                                "description": "The transaction ID to look up"
                            }
                        },
                        "required": ["transaction_id"]
                    }
                }
            }
        },
        {
            "toolSpec": {
                "name": "check_customer_history",
                "description": "Check customer spending history and patterns for a transaction",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "transaction_id": {
                                "type": "string",
                                "description": "The transaction ID to check history for"
                            }
                        },
                        "required": ["transaction_id"]
                    }
                }
            }
        },
    ]
}

# Map tool names to actual Python functions
TOOL_FUNCTIONS = {
    "lookup_transaction": scratch_lookup_transaction,
    "check_customer_history": scratch_check_history,
}

# Step 3: The ReAct Loop — this is the heart of every agent!
def react_agent_from_scratch(user_query: str, max_steps: int = 5):
    """A simple ReAct agent using raw Bedrock Converse API."""

    system_prompt = """You are a fraud investigation assistant at a financial institution.
When asked to investigate a transaction, use the available tools to gather information,
then provide your assessment. Always explain your reasoning step by step."""

    # Start the conversation with the user's query
    messages = [{"role": "user", "content": [{"text": user_query}]}]

    print(f"{'='*60}")
    print(f"USER: {user_query}")
    print(f"{'='*60}")

    for step in range(max_steps):
        # Call Bedrock with tool configuration
        # Uses SageMaker execution role for auth — no credentials needed
        response = bedrock_runtime.converse(
            modelId=BEDROCK_MODEL_ID,
            messages=messages,
            system=[{"text": system_prompt}],
            toolConfig=TOOL_CONFIG,
        )

        # Get the assistant's response
        output = response['output']['message']
        stop_reason = response['stopReason']

        # Add assistant response to conversation history
        messages.append(output)

        # Check if the model wants to use a tool
        if stop_reason == "tool_use":
            # Process each tool call in the response
            tool_results = []
            for content_block in output['content']:
                if 'toolUse' in content_block:
                    tool_call = content_block['toolUse']
                    tool_name = tool_call['name']
                    tool_input = tool_call['input']
                    tool_id = tool_call['toolUseId']

                    print(f"\n🔧 STEP {step + 1} — ACTION: {tool_name}({tool_input})")

                    # Execute the tool function
                    func = TOOL_FUNCTIONS[tool_name]
                    result = func(**tool_input)
                    print(f"   OBSERVATION: {result[:150]}...")

                    tool_results.append({
                        "toolResult": {
                            "toolUseId": tool_id,
                            "content": [{"text": result}]
                        }
                    })

            # Feed tool results back to the model
            messages.append({"role": "user", "content": tool_results})

        elif stop_reason == "end_turn":
            # Agent is done reasoning — extract final answer
            final_text = ""
            for content_block in output['content']:
                if 'text' in content_block:
                    final_text += content_block['text']

            print(f"\n{'='*60}")
            print(f"FINAL ANSWER (after {step + 1} steps):")
            print(f"{'='*60}")
            print(final_text)
            return final_text

    print("\n⚠️ Agent reached max steps without concluding.")
    return None

# Run it! Watch the agent think and act step by step.
result = react_agent_from_scratch("Investigate transaction TXN-001. Is it fraud?")

> **Think About It**: Look at the code above. The agent made multiple API calls to
> Bedrock — one for each "think + act" step. If you were building a production fraud
> system processing 10,000 transactions per day, what would the cost and latency
> implications be? How might you decide which transactions deserve full agent
> investigation vs. a quick classification (like what we built in Week 14)?

# Section 2: LangChain Agents with Custom Tools

Building agents from scratch is educational, but in practice we use frameworks like
**LangChain** to handle the boilerplate. LangChain provides:

- **`ChatBedrockConverse`**: Wraps the Bedrock API as a LangChain chat model
- **`@tool` decorator**: Turns any Python function into a tool the agent can call
- **`create_react_agent`**: Creates a complete ReAct agent with one line of code
- **`MemorySaver`**: Adds conversation memory (we'll use this in Section 3)

Let's rebuild our fraud investigation agent — this time with LangChain.

## Step 1: Initialize the LLM

First, we wrap Bedrock as a LangChain chat model. On SageMaker, `ChatBedrockConverse`
automatically picks up credentials from the execution role — no manual configuration:

```python
llm = ChatBedrockConverse(
    model=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.1,
    max_tokens=1024,
)
```

In [ ]:
# =============================================================================
# DEMO: Initialize ChatBedrockConverse
# =============================================================================
# ChatBedrockConverse wraps Bedrock's Converse API as a LangChain chat model.
# On SageMaker, it automatically uses the execution role for authentication —
# no manual credential setup needed.

llm = ChatBedrockConverse(
    model=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.1,         # Low temperature for consistent fraud analysis
    max_tokens=1024,         # Enough for detailed analysis
)

# Quick test — same as calling bedrock_runtime.converse() but much simpler
response = llm.invoke("What are three signs of a fraudulent transaction?")
print("LLM Response:")
print(response.content)
print(f"\nType: {type(response)}")
print(f"Usage: {response.usage_metadata}")

In [ ]:
# =============================================================================
# DEMO: Create Custom Tools with @tool Decorator
# =============================================================================
# The @tool decorator turns a Python function into a LangChain tool.
# The function's docstring becomes the tool description — this is how the LLM
# decides WHEN and HOW to use each tool. Good docstrings = smart agents!

@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID.

    Use this tool when you need to find information about a specific transaction,
    including the amount, merchant, type, time, location, and description.
    The transaction_id should be in the format TXN-XXX (e.g., TXN-001).
    """
    txn = TRANSACTION_DATABASE.get(transaction_id)
    if txn:
        return json.dumps(txn, indent=2)
    return f"Transaction {transaction_id} not found in database."


@tool
def check_customer_history(transaction_id: str) -> str:
    """Check customer spending history and patterns for a given transaction.

    Use this tool to understand a customer's normal behavior — their average
    monthly spending, whether they've made international transactions before,
    their account age, typical active hours, and whether they've been flagged before.
    """
    history = CUSTOMER_HISTORY.get(transaction_id)
    if history:
        return json.dumps(history, indent=2)
    return f"No customer history found for {transaction_id}."


@tool
def calculate_risk_score(amount: float, avg_monthly_spend: float,
                         is_international: bool, is_unusual_hour: bool,
                         is_new_merchant: bool) -> str:
    """Calculate a fraud risk score (0-100) based on transaction characteristics.

    Higher scores indicate higher fraud risk. Uses a weighted formula:
    - Amount ratio (transaction amount / average monthly spend): 30% weight
    - International transaction with no history: 25% weight
    - Unusual hour (outside typical pattern): 20% weight
    - New merchant (not in customer history): 15% weight
    - Base risk: 10%
    """
    # Simple weighted risk calculation
    amount_ratio = min(amount / max(avg_monthly_spend, 1), 5.0)  # Cap at 5x
    amount_score = min(amount_ratio * 20, 100)  # Scale to 0-100

    intl_score = 90 if is_international else 0
    hour_score = 70 if is_unusual_hour else 0
    merchant_score = 50 if is_new_merchant else 0

    # Weighted combination
    risk_score = (
        amount_score * 0.30 +
        intl_score * 0.25 +
        hour_score * 0.20 +
        merchant_score * 0.15 +
        10  # Base risk
    )

    risk_level = "LOW" if risk_score < 30 else "MEDIUM" if risk_score < 60 else "HIGH"

    return json.dumps({
        "risk_score": round(risk_score, 1),
        "risk_level": risk_level,
        "components": {
            "amount_ratio_score": round(amount_score * 0.30, 1),
            "international_score": round(intl_score * 0.25, 1),
            "unusual_hour_score": round(hour_score * 0.20, 1),
            "new_merchant_score": round(merchant_score * 0.15, 1),
            "base_risk": 10
        }
    }, indent=2)


@tool
def check_fraud_policy(policy_type: str) -> str:
    """Look up a specific fraud prevention policy by type.

    Available policy types:
    - international_first_time: Rules for first-time international transactions
    - unusual_hours: Rules for transactions at unusual times
    - velocity_check: Rules for rapid successive transactions
    - geographic_mismatch: Rules for location anomalies
    - amount_threshold: Rules for unusually large transactions
    - new_payee_large_transfer: Rules for large transfers to new recipients
    - card_testing_pattern: Rules for potential card testing

    If you're not sure which policy applies, try 'all' to see all policies.
    """
    if policy_type == "all":
        return json.dumps(FRAUD_POLICIES, indent=2)

    policy = FRAUD_POLICIES.get(policy_type)
    if policy:
        return json.dumps({"policy_type": policy_type, "rule": policy}, indent=2)
    return f"Policy '{policy_type}' not found. Available: {list(FRAUD_POLICIES.keys())}"


# Show what we created
tools = [lookup_transaction, check_customer_history, calculate_risk_score, check_fraud_policy]
print("Tools created:")
for t in tools:
    print(f"  - {t.name}: {t.description[:80]}...")
print(f"\nTotal: {len(tools)} tools available for our agent")

In [ ]:
# =============================================================================
# DEMO: Create a ReAct Agent with LangChain + LangGraph
# =============================================================================
# create_react_agent() does all the plumbing we did manually in Cell 6:
# - Connects the LLM to the tools
# - Implements the think -> act -> observe loop
# - Handles tool calling and response formatting
#
# One line of code replaces ~60 lines of our from-scratch agent!

fraud_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a senior fraud analyst at a major financial institution. "
           "When investigating a transaction, always:\n"
           "1. Look up the transaction details first\n"
           "2. Check the customer's spending history\n"
           "3. Calculate a risk score based on what you found\n"
           "4. Check relevant fraud policies\n"
           "5. Provide a clear VERDICT (FRAUD / LEGITIMATE / NEEDS REVIEW) "
           "with detailed reasoning\n\n"
           "Be thorough but concise. Cite specific evidence from the tools."
)

print("✅ Fraud investigation agent created!")
print(f"   LLM: {BEDROCK_MODEL_ID}")
print(f"   Tools: {[t.name for t in tools]}")

In [ ]:
# =============================================================================
# DEMO: Run the Agent — Watch It Think!
# =============================================================================
# We'll use stream() to see each step of the agent's reasoning in real time.
# TXN-005: Customer disputes a $2,100 jewelry purchase in Miami while they
# were confirmed to be in Chicago. Classic geographic mismatch fraud.

print("=" * 70)
print("FRAUD INVESTIGATION: TXN-005")
print("=" * 70)

# Stream the agent's reasoning step by step
for step in fraud_agent.stream(
    {"messages": [HumanMessage(content="Investigate transaction TXN-005. Is it fraud or legitimate? Provide your complete analysis.")]},
    stream_mode="updates",
):
    # Each step is a dict with the node name and its output
    for node_name, node_output in step.items():
        if node_name == "agent":
            # The LLM's thinking/response
            for msg in node_output.get("messages", []):
                if hasattr(msg, 'tool_calls') and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"\n🔧 CALLING TOOL: {tc['name']}")
                        print(f"   Input: {json.dumps(tc['args'], indent=2)}")
                elif hasattr(msg, 'content') and msg.content:
                    print(f"\n📋 AGENT RESPONSE:")
                    print(msg.content)

        elif node_name == "tools":
            # Tool execution results
            for msg in node_output.get("messages", []):
                print(f"\n📊 TOOL RESULT ({msg.name}):")
                content = msg.content if isinstance(msg.content, str) else str(msg.content)
                print(f"   {content[:200]}{'...' if len(content) > 200 else ''}")

print(f"\n{'=' * 70}")
print("Investigation complete!")

In [ ]:
# =============================================================================
# DEMO: Run Agent on a Legitimate Transaction (for contrast)
# =============================================================================
# TXN-008: A routine $67.23 grocery purchase at Whole Foods.
# The agent should recognize this as normal behavior.

print("=" * 70)
print("FRAUD INVESTIGATION: TXN-008 (Expected: Legitimate)")
print("=" * 70)

# Use invoke() for simpler output (no streaming)
result = fraud_agent.invoke(
    {"messages": [HumanMessage(content="Investigate transaction TXN-008. Is it fraud or legitimate?")]}
)

# Print the final response
final_message = result["messages"][-1]
print(f"\n📋 FINAL VERDICT:")
print(final_message.content)
print(f"\n💡 The agent used {len(result['messages']) - 1} messages (including tool calls)")

# Lab 1: Build a Fraud Investigation Agent (15 minutes)

## Your Mission

You'll create your own fraud investigation agent with a **new custom tool** and test
it on multiple transactions.

## Instructions

### Step 1: Create a new tool — `get_similar_transactions`

Create a tool that, given a transaction type (e.g., "wire_transfer", "atm_withdrawal"),
returns all transactions of that type from our database. This helps the agent see
patterns across similar transactions.

The tool should:
- Accept a `transaction_type` parameter (string)
- Search `TRANSACTION_DATABASE` for transactions matching that type
- Return a JSON string with the matching transactions (id, amount, and description snippet)
- If no matches, return a helpful message

### Step 2: Create your agent

Create a `create_react_agent` with:
- The `llm` (ChatBedrockConverse) from earlier
- All 4 existing tools PLUS your new `get_similar_transactions` tool
- A system prompt that instructs the agent to also check for patterns in similar transactions

### Step 3: Test your agent

Investigate these transactions and verify the agent uses your new tool:
- `TXN-009` (expected: fraud — password change + large transfer)
- `TXN-004` (expected: legitimate — normal Amazon purchase)

## Hints

- The `@tool` decorator makes any function into a tool
- The docstring is crucial — it tells the LLM when to use the tool
- `transaction_type` values in our database: "wire_transfer", "atm_withdrawal", "online_purchase", "in_store", "subscription", "direct_deposit"
- Use a list comprehension to filter `TRANSACTION_DATABASE.values()`

## Homework Extension

After class, try:
1. Add a `flag_for_review` tool that "flags" a transaction (prints a message simulating a database update)
2. Test your agent on TXN-003, TXN-007, and TXN-010
3. Count how many API calls each investigation requires

In [ ]:
# =============================================================================
# LAB 1: Build a Fraud Investigation Agent
# =============================================================================

# Step 1: Create the get_similar_transactions tool
@tool
def get_similar_transactions(transaction_type: str) -> str:
    """Find all transactions of a given type to identify patterns.

    Use this tool when you want to see if a transaction type has been
    associated with fraud before. Helps identify patterns across
    similar transactions.

    Available types: wire_transfer, atm_withdrawal, online_purchase,
    in_store, subscription, direct_deposit
    """
    similar = None  # YOUR CODE

    if similar:
        return json.dumps(similar, indent=2)
    return f"No transactions of type '{transaction_type}' found."


# Step 2: Create your agent with all 5 tools
all_tools = None  # YOUR CODE

my_agent = None  # YOUR CODE


# Step 3: Test on TXN-009 (expected: fraud — password change + large wire transfer)
result_009 = None  # YOUR CODE

# Print the final verdict
print("TXN-009 Verdict:")
print("=" * 50)
# YOUR CODE — print result_009["messages"][-1].content


# Step 4: Test on TXN-004 (expected: legitimate — normal Amazon purchase)
result_004 = None  # YOUR CODE

print("\nTXN-004 Verdict:")
print("=" * 50)
# YOUR CODE — print result_004["messages"][-1].content

# Section 3: Memory — Agents That Remember

So far, our agent treats each investigation as independent. But real fraud analysts
have **conversations**: they might investigate one transaction, then say "What about
the other wire transfer?" — and the agent should know what "other" refers to.

## How Memory Works in LangGraph

LangGraph provides a **checkpointing** system that saves the agent's state after
each interaction. When you use the same `thread_id`, the agent picks up where it
left off:

```
Thread "investigation-42":
  Turn 1: "Investigate TXN-001" -> Agent investigates, saves state
  Turn 2: "Compare it to TXN-002" -> Agent remembers TXN-001, adds TXN-002 context
  Turn 3: "Which one is riskier?" -> Agent compares both from memory
```

This replaces the older `ConversationBufferMemory` pattern (now deprecated in LangChain).
The LangGraph approach is more robust and naturally supports advanced features like
time travel and branching — which we'll explore in Week 16.

The key ingredient is **`MemorySaver`** as the checkpointer, and a **`thread_id`** in
the config to identify each conversation:

```python
memory = MemorySaver()
agent = create_react_agent(model=llm, tools=tools, checkpointer=memory)

# Same thread_id = same conversation
config = {"configurable": {"thread_id": "my-investigation"}}
agent.invoke({"messages": [HumanMessage(content="...")]}, config=config)
```

In [ ]:
# =============================================================================
# DEMO: Agent with Conversation Memory (MemorySaver)
# =============================================================================
import warnings
warnings.filterwarnings("ignore", message=".*attempt to write a readonly database.*")
warnings.filterwarnings("ignore", message=".*history saving thread.*")

# Step 1: Create a MemorySaver checkpointer
# This stores the full conversation state in memory (for production, you'd
# use a persistent backend like a database)
memory = MemorySaver()

# Step 2: Create agent WITH memory
agent_with_memory = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a senior fraud analyst conducting investigations. "
           "Remember the context of our conversation — if I refer to "
           "'the previous transaction' or 'compare them', use your memory "
           "of earlier investigations in this thread.",
    checkpointer=memory,  # This enables memory!
)

# Step 3: Use a thread_id to maintain conversation state
# Think of this as a "case file" — everything in the same thread_id
# belongs to the same investigation session
thread_config = {"configurable": {"thread_id": "investigation-demo"}}

# Turn 1: Investigate first transaction
print("=" * 60)
print("TURN 1: Investigate TXN-001")
print("=" * 60)

result1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="Investigate TXN-001.")]},
    config=thread_config,  # Same thread_id across all turns
)
print(result1["messages"][-1].content[:500])

In [ ]:
# =============================================================================
# DEMO: Multi-Turn Conversation (Agent Remembers!)
# =============================================================================

# Turn 2: Reference previous investigation — the agent should remember TXN-001
print("=" * 60)
print("TURN 2: Follow-up question")
print("=" * 60)

result2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(
        content="Now investigate TXN-002. Is it similar to the previous one?"
    )]},
    config=thread_config,  # SAME thread_id — agent remembers Turn 1!
)
print(result2["messages"][-1].content[:500])

# Turn 3: Ask for comparison (agent should remember BOTH investigations)
print("\n" + "=" * 60)
print("TURN 3: Comparison request")
print("=" * 60)

result3 = agent_with_memory.invoke(
    {"messages": [HumanMessage(
        content="Compare the risk levels of both transactions. Which needs more attention?"
    )]},
    config=thread_config,  # Still same thread — full context preserved
)
print(result3["messages"][-1].content[:500])

# Show conversation length — proof that memory is accumulating
all_messages = result3["messages"]
print(f"\n💡 Conversation so far: {len(all_messages)} messages across 3 turns")
print(f"   The agent has full context of everything we discussed!")

In [ ]:
# =============================================================================
# DEMO: New Thread = No Memory
# =============================================================================
# Using a DIFFERENT thread_id starts a completely fresh conversation.
# This is how you handle multiple concurrent investigations.

new_thread = {"configurable": {"thread_id": "different-investigation"}}

result_fresh = agent_with_memory.invoke(
    {"messages": [HumanMessage(
        content="What did we discuss about the previous transaction?"
    )]},
    config=new_thread,  # DIFFERENT thread_id — no memory of previous turns!
)

print("Response with NEW thread (no memory):")
print(result_fresh["messages"][-1].content[:300])
print(f"\n💡 Different thread_id = different conversation = no shared memory.")
print(f"   This is how you isolate concurrent investigations.")

> **Think About It**: Memory means the agent accumulates ALL previous messages.
> After 50 turns, the conversation context could be enormous. What problems might
> this cause? (Think about cost, latency, context window limits.) In Week 16,
> we'll learn about LangGraph's state management which lets you trim, summarize,
> or selectively store conversation history. For now, consider: in a real
> fraud investigation system, how long should an agent "remember" a case?

# Lab 2: Conversational Fraud Analyst (15 minutes)

## Your Mission

Build an agent with memory that can conduct a multi-step fraud investigation
across several turns of conversation.

## Instructions

### Step 1: Create an agent with memory

- Use `MemorySaver()` for the checkpointer
- Use all available tools (the 4 original tools — or 5 if you completed Lab 1)
- Set a system prompt that emphasizes thorough investigation and memory

### Step 2: Conduct a 3-turn investigation

Using the SAME thread_id:

1. **Turn 1**: "Investigate TXN-007 — I got an alert about suspicious small purchases."
2. **Turn 2**: "Check if there are other online purchases in our database. Is this a pattern?"
3. **Turn 3**: "Based on everything you've found, write a brief investigation summary with your recommendation."

### Step 3: Verify memory is working

- The agent should reference findings from previous turns
- Turn 3's summary should include information from Turns 1 and 2

## Expected Output

Your agent should identify TXN-007 as suspicious (multiple small purchases at unknown
merchants) and provide a coherent multi-turn investigation report.

## Homework Extension

After class, try:
1. Conduct a 5-turn investigation that covers TXN-007, TXN-003, and TXN-010
2. Ask the agent to identify common patterns across all three fraud cases
3. Have the agent draft a "fraud pattern alert" based on its findings

In [ ]:
# =============================================================================
# LAB 2: Conversational Fraud Analyst
# =============================================================================

# Step 1: Create agent with memory
lab_memory = None  # YOUR CODE

lab_agent = None  # YOUR CODE

# Use a unique thread_id for this investigation
lab_thread = {"configurable": {"thread_id": "lab-2-investigation"}}

# Step 2: Turn 1 — Initial alert
print("TURN 1: Initial Alert")
print("-" * 40)
turn1_result = None  # YOUR CODE

if turn1_result is not None:
    print(turn1_result["messages"][-1].content[:400])

# Step 3: Turn 2 — Pattern check (use SAME lab_thread!)
print("\nTURN 2: Pattern Check")
print("-" * 40)
turn2_result = None  # YOUR CODE

if turn2_result is not None:
    print(turn2_result["messages"][-1].content[:400])

# Step 4: Turn 3 — Summary (use SAME lab_thread!)
print("\nTURN 3: Investigation Summary")
print("-" * 40)
turn3_result = None  # YOUR CODE

if turn3_result is not None:
    print(turn3_result["messages"][-1].content[:400])

# Step 5: Verify memory — check total message count
if turn3_result is not None:
    print(f"\nTotal messages in thread: {len(turn3_result['messages'])}")
    print("✅ If Turn 3 references findings from Turns 1 and 2, memory is working!")

# Wrap-up: What We Learned Today

## Key Takeaways

1. **AI agents = LLM + tools + loop**: An agent reasons about what to do, calls
   tools to get information, observes results, and repeats until done.

2. **The ReAct pattern** (Reasoning + Acting) is the foundation of modern agents.
   We built one from scratch, then used LangChain to do it with much less code.

3. **Custom tools are just Python functions** decorated with `@tool`. The docstring
   is crucial — it's how the LLM decides when to use each tool.

4. **Memory via LangGraph checkpointing** lets agents maintain context across
   turns. Same `thread_id` = same conversation; different `thread_id` = fresh start.

## Connection to Week 16

Next week, we pivot to the **AWS production stack** for agents:

- **Strands Agents SDK** — AWS's open-source agent framework. Same concepts
  as LangChain (agents, `@tool`, memory) but with a simpler API and native
  AWS integration. You'll rebuild today's fraud agent in Strands and see how
  portable your knowledge is.
- **Multi-agent orchestration** — specialized agents (Triage, Investigation,
  Decision) that collaborate through the "agents-as-tools" pattern. Each agent
  becomes a callable tool for a supervisor agent.
- **Amazon Bedrock AgentCore** — the managed infrastructure for production agents.
  Persistent memory (survives restarts), gateway (MCP tool conversion), runtime
  (container hosting), identity, and observability. This is what Bread Financial
  will use in production.

**Key point**: everything you learned today — ReAct, tools, memory — transfers
directly. The concepts are framework-agnostic. Next week we just switch to the
toolkit you'll actually use at work.

# Homework & Optional Extensions

## Homework (Complete before next session)

### Extension 1: More Investigations
Investigate ALL 10 transactions in our database. Create a summary table showing
each transaction's verdict and risk score. Which ones did the agent get right?

### Extension 2: DistilBERT as a Tool (Connecting Week 14!)
Load your fine-tuned DistilBERT model from Week 14 and wrap it as an agent tool:

```python
from transformers import pipeline

# Load your fine-tuned model
classifier = pipeline("text-classification", model="./week14_fraud_model")

@tool
def classify_with_distilbert(transaction_description: str) -> str:
    """Use the fine-tuned DistilBERT model from Week 14 to classify
    a transaction as fraud or legitimate. Returns the model's prediction
    and confidence score."""
    result = classifier(transaction_description)[0]
    return json.dumps({
        "model": "DistilBERT (fine-tuned Week 14)",
        "prediction": result["label"],
        "confidence": round(result["score"], 4)
    })
```

Then add this tool to your agent and see how it incorporates the model's prediction
into its investigation. The agent now has BOTH reasoning (LLM) and a specialized
classifier (DistilBERT) — a hybrid architecture!

### Extension 3: Cost Tracking
Add a wrapper that counts Bedrock API calls per investigation. Calculate the
approximate cost per investigation using Claude Haiku pricing. Compare this to
the cost of running your fine-tuned DistilBERT from Week 14 (hint: $0 after training).

### Extension 4: Agent Debugging
When the agent makes a wrong decision, trace through its reasoning steps.
What tool calls did it make? What information did it miss? How would you
improve the tools or system prompt to fix it?

---

# Great Work Today!

You've completed Week 15 of the AI for Data Scientists Academy. You now know how
to build AI agents that can reason, use tools, and hold conversations. Next week,
we take these same concepts and apply them with **Strands Agents SDK** and
**Amazon Bedrock AgentCore** — building multi-agent fraud pipelines on the exact
production stack Bread Financial will use.